In [69]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [70]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = 'True'
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT")


In [71]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/langsmith/manage-prompts")


In [72]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts', 'title': 'Manage prompts - Docs by LangChain', 'description': 'Manage prompt versions, environments, and access controls in LangSmith.', 'language': 'en'}, page_content='Manage prompts - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage promptsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsChatModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsContext engineeringUse the Context Hu

In [73]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
document = text_splitter.split_documents(docs)


In [74]:
document

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts', 'title': 'Manage prompts - Docs by LangChain', 'description': 'Manage prompt versions, environments, and access controls in LangSmith.', 'language': 'en'}, page_content='Manage prompts - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage promptsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsChatModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsContext engineeringUse the Context Hu

In [75]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [76]:
from langchain_community.vectorstores import FAISS

vectorstoredb = FAISS.from_documents(
    documents=document,
    embedding=embeddings
)

In [77]:
vectorstoredb

In [78]:
query = "tell me about Prompt owners"
result= vectorstoredb.similarity_search(query)
result[0].page_content

'For more information on how to use prompts in code, refer to Managing prompts programmatically.\n\u200bPrompt owners\nThe prompt owners feature gives you fine-grained control over who can tag commits and delete a specific prompt. This is useful for production promotion flows where you want to limit which team members can promote a commit to an environment by assigning or moving tags.\n\u200bAccess modes\nEach prompt has two access modes, configured under Access and Permissions in the UI:\n\nWorkspace authorized users (default): any workspace user with the prompts:tag permission can create, update, and delete tags and delete the prompt.\nOwners only: only users added as prompt owners can create or update commit tags, promote commits to environments, and delete the prompt.\n\nLangSmith automatically adds the prompt creator as an owner.\n\u200bConfigure access and permissions'

In [79]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

llm = OllamaLLM(model="gemma:2b")

prompt = ChatPromptTemplate.from_template(
"""
Answer the question based only on the provided context.

<context>
{context}
</context>

Question: {input}
"""
)

document_chain = prompt | llm

In [80]:
document_chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the provided context.\n\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
| OllamaLLM(model='gemma:2b')

In [81]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"commit tags are labels ",
    
    "context":[Document(page_content="Commit tags are labels that reference a specific commit in your prompt’s version history. They help you mark significant versions and control which versions run in different environments. By referencing tags rather than commit IDs in your code, you can update which version is being used without modifying the code itself.")]
}
)

'Sure, based on the context, commit tags are labels that reference a specific commit in your prompt’s version history.'

In [82]:
vectorstoredb.as_retriever()

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024F2FF06390>, search_kwargs={})

In [83]:
retriever = vectorstoredb.as_retriever()

from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [84]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024F2FF06390>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the provided context.\n\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
            | OllamaLLM(model='gemma:2b')
  }), kwargs={}, config={'run_name': 'retrieval_chain'}, config_factories=[])

In [85]:
response = retrieval_chain.invoke({"input": "Commit tags for version control and environment management."})

response['answer']

'Sure, based on the context, commit tags are specific to prompt versioning and reference individual commits in a prompt’s history. They help to mark significant versions and control which versions run in different environments.'

In [87]:
response

{'input': 'Commit tags for version control and environment management.',
 'context': [Document(id='c11a0b54-01fa-4435-97d3-d91b0d7f77f0', metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts', 'title': 'Manage prompts - Docs by LangChain', 'description': 'Manage prompt versions, environments, and access controls in LangSmith.', 'language': 'en'}, page_content='\u200bCommit tags\nCommit tags are labels that reference a specific commit in your prompt’s version history. They help you mark significant versions and control which versions run in different environments. By referencing tags rather than commit IDs in your code, you can update which version is being used without modifying the code itself.\nEach tag references exactly one commit, though you can reassign a tag to point to a different commit.\nReserved tags: The staging and production tags are reserved for environment management and are not enabled in the freeform tag picker. Use the promotion flow to assign com